<h1><center>Практическое задание</center></h1>

1. Выбрать модель и набор данных для решения задачи ответов на вопросы
2. Обучить модель на предобработанных данных
3. Самостоятельно сделвть построцесс результатов для отображения в человекочитаемом формате
4. Собрать метрики качества модели

In [2]:
from datasets import load_dataset

ds = load_dataset("kuznetsoffandrey/sberquad")

In [3]:
ds

DatasetDict({
    train: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 45328
    })
    validation: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 5036
    })
    test: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 23936
    })
})

In [5]:
ds['train'][250]

{'id': 23417,
 'title': 'SberChallenge',
 'context': 'После смерти Сталина (1953) запрет на некоторых писателей был снят. Начиная с 1956 года, усилиями К. Паустовского, Ю. Олеши, И. Новикова и других, Грин был возвращён в литературу. Его произведения издавались миллионными тиражами. Получив стараниями друзей Грина гонорар за Избранное (1956), Нина Николаевна приехала в Старый Крым, с трудом отыскала заброшенную могилу мужа и выяснила, что дом, где умер Грин, перешёл к председателю местного исполкома и использовался как сарай и курятник[75]. В 1960 году, после нескольких лет борьбы за возвращение дома, Нина Николаевна открыла на общественных началах Музей Грина в Старом Крыму. Там она провела последние десять лет своей жизни, с пенсией 21 рубль (авторские права больше не действовали)[76]. В июле 1970 года был открыт также Музей Грина в Феодосии, а год спустя дом Грина в Старом Крыму тоже получил статус музея. Его открытие крымским обкомом КПСС увязывалось с конфликтом с Ниной Николаевно

In [7]:
from transformers import AutoTokenizer, AutoModelForQuestionAnswering


tokenizer = AutoTokenizer.from_pretrained("cointegrated/rubert-tiny2")
model = AutoModelForQuestionAnswering.from_pretrained("cointegrated/rubert-tiny2", device_map="auto")

Loading weights:   0%|          | 0/53 [00:00<?, ?it/s]

[transformers] BertForQuestionAnswering LOAD REPORT from: cointegrated/rubert-tiny2
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
qa_outputs.weight                          | MISSING    | 
qa_outputs.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were

In [8]:
tokenizer(
    'После смерти Сталина (1953) запрет на некоторых писателей был снят. Начиная с 1956 года, усилиями К. Паустовского, Ю. Олеши, И. Новикова и других, Грин был возвращён в литературу. Его произведения издавались миллионными тиражами.'
)

{'input_ids': [2, 2300, 6514, 41923, 12, 1238, 13, 32700, 548, 8904, 22243, 991, 29039, 18, 27873, 329, 1162, 735, 16, 45339, 290, 18, 83291, 13290, 1302, 16, 310, 18, 11450, 26430, 16, 288, 18, 77505, 320, 3725, 16, 36807, 991, 76210, 314, 44860, 18, 5971, 13387, 82642, 35335, 3092, 55339, 1854, 18, 3], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}

In [9]:
import numpy as np
import pandas as pd

df = pd.DataFrame([[len(tokenizer(i['context'])['input_ids']),
                    len(tokenizer(i['question'])['input_ids'])]
                    for i in ds['train']], 
                  columns=['context_len', 'question_len'])
df.describe()

,context_len,question_len
count,45328.000000,45328.000000
mean,152.213841,14.144789
std,56.088350,6.021840
min,59.000000,4.000000
25%,115.000000,10.000000
50%,139.000000,13.000000
75%,176.000000,16.000000
max,1670.000000,122.000000


In [10]:
def prepare_train_features(examples, max_length, doc_stride, pad_on_right):
    examples['question'] = [q.lstrip() for q in examples['question']]
    tokenized_examples = tokenizer(
        examples['question' if pad_on_right else 'context'],
        examples['context' if pad_on_right else 'question'],
        truncation='only_second' if pad_on_right else 'only_first',
        max_length=max_length,
        stride=doc_stride,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        return_token_type_ids=True,
        padding='max_length',
    )
    sample_mapping = tokenized_examples.pop('overflow_to_sample_mapping')
    offset_mapping = tokenized_examples.pop('offset_mapping')

    tokenized_examples['start_positions'] = []
    tokenized_examples['end_positions'] = []

    for i, offsets in enumerate(offset_mapping):
        input_ids = tokenized_examples['input_ids'][i]
        cls_index = input_ids.index(tokenizer.cls_token_id)

        sequence_ids = tokenized_examples.sequence_ids(i)

        sample_index = sample_mapping[i]
        answers = examples['answers'][sample_index]

        if len(answers['answer_start']) == 0:   
            tokenized_examples['start_positions'].append(cls_index)
            tokenized_examples['end_positions'].append(cls_index)
        else:
            start_char = answers['answer_start'][0]
            end_char = start_char + len(answers['text'][0])

            token_start_index = 0
            while sequence_ids[token_start_index] != (1 if pad_on_right else 0):
                token_start_index += 1

            token_end_index = len(input_ids) - 1
            while sequence_ids[token_end_index] != (1 if pad_on_right else 0):
                token_end_index -= 1
            
            if not(offsets[token_start_index][0] <= start_char and 
                   offsets[token_end_index][1] >= end_char):
                tokenized_examples['start_positions'].append(cls_index)
                tokenized_examples['end_positions'].append(cls_index)
            else:
                while token_start_index < len(offsets) and offsets[token_start_index][0] <= start_char:
                    token_start_index += 1
                tokenized_examples['start_positions'].append(token_start_index - 1)
                
                while offsets[token_end_index][1] >= end_char:
                    token_end_index -= 1
                tokenized_examples['end_positions'].append(token_end_index - 1)
    return tokenized_examples

In [11]:
pad_on_right = tokenizer.padding_side == 'right'
max_length = 256
doc_stride = 64

features = prepare_train_features(ds['train'][150:155], max_length, doc_stride, pad_on_right)

tokenized_datasets = ds.map(prepare_train_features, batched=True,
                            fn_kwargs={'max_length': max_length, 'doc_stride': doc_stride, 'pad_on_right': pad_on_right},
                            remove_columns=ds['train'].column_names)

In [13]:
from transformers import TrainingArguments, Trainer, default_data_collator


model_checkpoint = "cointegrated/rubert-tiny2"
batch_size = 32
squad_v2 = False
model_name = model_checkpoint.split('/')[-1]
data_collator = default_data_collator
args = TrainingArguments(
    f'{model_name}-finutuned-squad',
    eval_strategy='epoch',
    learning_rate=2e-5,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    num_train_epochs=10,
    weight_decay=.01
)

print(model)


BertForQuestionAnswering(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(83828, 312, padding_idx=0)
      (position_embeddings): Embedding(2048, 312)
      (token_type_embeddings): Embedding(2, 312)
      (LayerNorm): LayerNorm((312,), eps=1e-12, elementwise_affine=True, bias=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-2): 3 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=312, out_features=312, bias=True)
              (key): Linear(in_features=312, out_features=312, bias=True)
              (value): Linear(in_features=312, out_features=312, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=312, out_features=312, bias=True)
              (LayerNorm): LayerNorm((312,), eps=1

In [ ]:

trainer = Trainer(
    model,
    args,
    train_dataset=tokenized_datasets['train'],
    eval_dataset=tokenized_datasets['validation'],
    data_collator=data_collator,
    processing_class=tokenizer
)

trainer.train()

In [ ]:
# trainer.save_model('test-rubert-trained')


# Подгрузка модели

path_to_model = "D:/MIREA/DPO/Learning/Practic2_20/rubert-tiny2-finutuned-squad\checkpoint-15100"

tokenizer = AutoTokenizer.from_pretrained(path_to_model, use_fast=True)
model = AutoModelForQuestionAnswering.from_pretrained(path_to_model)


trainer = Trainer(
    model,
    args,
    train_dataset=tokenized_datasets['train'],
    eval_dataset=tokenized_datasets['validation'],
    data_collator=data_collator,
    processing_class=tokenizer
)

Loading weights:   0%|          | 0/55 [00:00<?, ?it/s]

In [20]:
import torch


def prepare_validation_features(examples, max_length, doc_stride, pad_on_right):
    examples['question'] = [q.lstrip() for q in examples['question']]
    tokenized_examples = tokenizer(
        examples['question' if pad_on_right else 'context'],
        examples['context' if pad_on_right else 'question'],
        truncation='only_second' if pad_on_right else 'only_first',
        max_length=max_length,
        stride=doc_stride,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        return_token_type_ids=True,
        padding='max_length',
    )
    sample_mapping = tokenized_examples.pop('overflow_to_sample_mapping')
    tokenized_examples['example_id'] = []

    for i in range(len(tokenized_examples['input_ids'])):
        sequence_ids = tokenized_examples.sequence_ids(i)
        context_index = 1 if pad_on_right else 0
        sample_index = sample_mapping[i]
        tokenized_examples['example_id'].append(examples['id'][sample_index])
        tokenized_examples['offset_mapping'][i] = [
            (o if sequence_ids[k] == context_index else None)
            for k, o in enumerate(tokenized_examples['offset_mapping'][i])

        ]
    return tokenized_examples


for batch in trainer.get_eval_dataloader():
    break
batch = {k: v.to(trainer.args.device) for k, v in batch.items()}
with torch.inference_mode():
    output = trainer.model(**batch)
output.keys()

n_best_size = 20
start_logits = output.start_logits[0].cpu().numpy()
end_logits = output.end_logits[0].cpu().numpy()
start_indexes = np.argsort(start_logits)[-1: -n_best_size - 1: -1].tolist()
end_indexes = np.argsort(end_logits)[-1: -n_best_size - 1: -1].tolist()
valid_answers = []
for start_index in start_indexes:
    for end_index in end_indexes:
        if start_index <= end_index:
            valid_answers.append(
                {
                'score': start_logits[start_index] + end_logits[end_index],
                'text': ''
                }
            )

validation_features = ds['validation'].map(
    prepare_validation_features,
    fn_kwargs={'max_length': max_length, 'doc_stride': doc_stride, 'pad_on_right': pad_on_right},
    batched=True,
    remove_columns=ds['validation'].column_names
)

raw_predicitons = trainer.predict(validation_features)
print('Raw predictions shape: ', raw_predicitons.predictions[1].shape)

Map:   0%|          | 0/5036 [00:00<?, ? examples/s]

Raw predictions shape:  (5372, 256)


In [21]:
validation_features.set_format(type=validation_features.format['type'],
                               columns=list(validation_features.features.keys()))
max_answer_length = 30

start_logits = output.start_logits[5].cpu().numpy()
end_logits = output.end_logits[5].cpu().numpy()
offset_mapping = validation_features[5]['offset_mapping']
context = ds['validation'][5]['context']
start_indexes = np.argsort(start_logits)[-1: -n_best_size - 1: -1].tolist()
end_indexes = np.argsort(end_logits)[-1: -n_best_size - 1: -1].tolist()
valid_answers = []
for start_index in start_indexes:
    for end_index in end_indexes:
        if (
            start_index >= len(offset_mapping)
            or end_index >= len(offset_mapping)
            or offset_mapping[start_index] is None
            or offset_mapping[end_index] is None
        ):
            continue
        if end_index < start_index or end_index - start_index + 1 > max_answer_length:
            continue
        if start_index <= end_index:
            start_char = offset_mapping[start_index][0]
            end_char = offset_mapping[end_index][1]
            valid_answers.append(
                {
                    'score': start_logits[start_index] + end_logits[end_index],
                    'text': context[start_char: end_char]
                } 
            )

valid_answers = sorted(valid_answers, key = lambda x: x['score'], reverse = True)[:n_best_size]
valid_answers

[{'score': np.float32(6.3607445),
  'text': 'как простая последовательность гласных или дифтонгов. Как и монофтонги, дифтонги делятся на две группы: оральные и'},
 {'score': np.float32(6.1212034), 'text': 'оральные и'},
 {'score': np.float32(5.995879),
  'text': 'простая последовательность гласных или дифтонгов. Как и монофтонги, дифтонги делятся на две группы: оральные и'},
 {'score': np.float32(5.2484226),
  'text': 'как простая последовательность гласных или диф'},
 {'score': np.float32(4.8835573),
  'text': 'простая последовательность гласных или диф'},
 {'score': np.float32(4.3393745), 'text': ': оральные и'},
 {'score': np.float32(3.8562245),
  'text': 'как простая последовательность гласных или дифтонгов. Как и монофтонги, дифтонги делятся на две группы: оральные и наз'},
 {'score': np.float32(3.6166835), 'text': 'оральные и наз'},
 {'score': np.float32(3.4913592),
  'text': 'простая последовательность гласных или дифтонгов. Как и монофтонги, дифтонги делятся на две группы: орал

In [72]:
ds['validation'][5]

{'id': 75698,
 'title': 'SberChallenge',
 'context': 'Нисходящие дифтонги со слабым гласным /i/ и /u/ часто встречаются в португальском. Восходящие дифтонги со слабым гласным /i/ и /u/ встречаются реже, и мноие из них могут быть квалифицированы как хиатусы (напр., fé-rias [как дифтонг] и fé-ri-as [как хиатус]: оба варианта анализа грамматически допустимы). Разница между восходящими дифтонгом и хиатусом не фонематична; первые обычно встречаются в в разговорной речи, а вторые — в аккуратном произношении. Существуют также трифтонги (напр., /uai/ как в слове Paraguai, или /iau/ как в слове miau). Могут встречаться и более длинные последовательности гласных, но они квалифицируются как простая последовательность гласных или дифтонгов. Как и монофтонги, дифтонги делятся на две группы: оральные и назальные.',
 'question': 'На какие группы классифицируют дифтонги?',
 'answers': {'text': ['оральные и назальные'], 'answer_start': [735]}}

In [1]:
ds

NameError: name 'ds' is not defined